# 🏀 バスケ試合動画の選手を「色分け＆追跡」する（YOLO + SAM2）

note記事「[バスケの試合動画をAIに見せたら、選手が勝手に色分け＆追跡された](https://note.com/hirosuke_0520/n/n85d57196fffd)」の内容を
**Google Colab の無料GPU** で再現するノートブックです。

このノートブックは **1試合分の長い動画でもメモリ落ち（OOM）しない逐次処理** に対応しています。

**処理の流れ**

1. **パス1 … 検出＆追跡**: YOLO11 で選手を検出、ByteTrack で各選手にIDを付与（この段階では画像を溜め込まず、検出データだけ収集）
2. **チーム分け**: 各選手のユニフォーム色（LAB色空間）でKMeansクラスタリング。ベンチ/観客はサイズと動き量で除外
3. **パス2 … 描画**: 動画をディスクから1フレームずつ読み直し、チーム色の枠を描いて書き出し（メモリ一定）
4. **CSV出力**: フレームごとの選手位置を `detections.csv` に保存（→ 個人成績・戦術分析の土台）
5. **（任意）SAM2 で個人の精密追跡**

> 💡 **使い方**: 「ランタイム → ランタイムのタイプを変更 → **GPU (T4)**」にしてから、上のセルから順に実行してください。


## 1. 環境セットアップ

In [ ]:
!nvidia-smi -L || echo "⚠️ GPUが見つかりません。ランタイム → ランタイムのタイプを変更 → GPU を選択してください。"


In [ ]:
%pip install -q "ultralytics>=8.3.0" scikit-learn
print("✅ インストール完了")


In [ ]:
import cv2, csv, math
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else "cpu"


## 2. 解析する動画を用意する

- **A. 自分の動画をアップロード**: 下のセルをそのまま実行してファイルを選択
- **B. サンプルで試す**: `USE_SAMPLE = True`


In [ ]:
USE_SAMPLE = False  # サンプル動画を使う場合は True

VIDEO_PATH = None
if USE_SAMPLE:
    import urllib.request
    sample_url = "https://media.roboflow.com/supervision/video-examples/basketball-1.mp4"
    VIDEO_PATH = "input.mp4"
    try:
        urllib.request.urlretrieve(sample_url, VIDEO_PATH)
        print("✅ サンプル動画を取得:", VIDEO_PATH)
    except Exception as e:
        print("⚠️ サンプル取得に失敗。手動アップロードに切替:", e); USE_SAMPLE = False

if not USE_SAMPLE:
    from google.colab import files
    print("動画ファイル(mp4)を選択してください…")
    uploaded = files.upload()
    VIDEO_PATH = list(uploaded.keys())[0]
    print("✅ アップロード:", VIDEO_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
N_TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FRAME_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); FRAME_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"📹 {VIDEO_PATH}: {FRAME_W}x{FRAME_H}, {FPS:.1f}fps, {N_TOTAL}フレーム (~{N_TOTAL/max(FPS,1):.1f}秒)")


### （任意）動画を短く切ってから試す

**1試合分をそのまま流してもメモリは大丈夫**な作りですが、初回の動作確認は短い方が速いです。
`TRIM_SECONDS` に秒数を入れると先頭だけ切り出します。**全部やるときは `None` のままでOK**。


In [ ]:
TRIM_SECONDS = None  # 例: 30 で先頭30秒。None なら動画全体

if TRIM_SECONDS:
    trimmed = "trimmed.mp4"
    !ffmpeg -y -i "{VIDEO_PATH}" -t {TRIM_SECONDS} -c:v libx264 -an "{trimmed}" -loglevel error
    VIDEO_PATH = trimmed
    cap = cv2.VideoCapture(VIDEO_PATH)
    FPS = cap.get(cv2.CAP_PROP_FPS)
    N_TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    FRAME_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); FRAME_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f"✂️ 先頭{TRIM_SECONDS}秒に切出し → {VIDEO_PATH} ({N_TOTAL}フレーム)")
else:
    print("トリミングなし（動画全体を使用）")


## 3. パス1: 選手の検出＆追跡（YOLO11 + ByteTrack）

動画をストリーム処理し、**各フレームの検出（フレーム番号・ID・bbox）** と **各IDのユニフォーム色サンプル** だけを集めます。
画像そのものは保持しないので、長い動画でもメモリは増えません。


In [ ]:
from ultralytics import YOLO

# 長い動画で速度が欲しければ "yolo11n.pt"、精度重視なら "yolo11x.pt"
det_model = YOLO("yolo11m.pt")
PERSON_CLASS = 0
CONF = 0.35

track_boxes  = defaultdict(list)  # tid -> [(frame, x1,y1,x2,y2), ...]
track_colors = defaultdict(list)  # tid -> [LAB色サンプル, ...]（最大MAX_COLOR件）
MAX_COLOR = 60

def torso_lab(frame, box):
    """bboxの上半身中央（ユニフォーム）の代表色をLABで返す"""
    x1, y1, x2, y2 = map(int, box)
    bw, bh = x2 - x1, y2 - y1
    cx1, cx2 = x1 + int(bw * 0.28), x2 - int(bw * 0.28)
    cy1, cy2 = y1 + int(bh * 0.18), y1 + int(bh * 0.45)
    crop = frame[max(cy1, 0):cy2, max(cx1, 0):cx2]
    if crop.size == 0:
        return None
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    return np.median(lab, axis=0)  # [L, a, b]

results = det_model.track(
    source=VIDEO_PATH, stream=True, persist=True,
    classes=[PERSON_CLASS], conf=CONF, tracker="bytetrack.yaml",
    device=DEVICE, verbose=False,
)

n_seen = 0
for f_idx, r in enumerate(results):
    n_seen += 1
    if r.boxes is None or r.boxes.id is None:
        continue
    frame = r.orig_img
    boxes = r.boxes.xyxy.cpu().numpy()
    ids = r.boxes.id.cpu().numpy().astype(int)
    for box, tid in zip(boxes, ids):
        track_boxes[tid].append((f_idx, *[float(v) for v in box]))
        if len(track_colors[tid]) < MAX_COLOR:
            c = torso_lab(frame, box)
            if c is not None:
                track_colors[tid].append(c)
    if f_idx % 300 == 0:
        print(f"  ...{f_idx}/{N_TOTAL} フレーム処理")

print(f"✅ パス1完了: {n_seen}フレーム / 検出IDユニーク数 = {len(track_boxes)}")


## 4. 選手フィルタ ＋ チーム分け

- **ベンチ・観客・審判の除外**: ①出現フレーム数が少ない ②枠が小さい（遠い/座っている）③ほとんど動かない、を外れ値として除外
- **チーム分け**: 各選手の代表ユニフォーム色（LAB中央値）で KMeans(2)。LABの明度Lが効くので「白 vs 紺」のような明暗差に強い

しきい値（`MIN_FRAMES` / `MIN_HEIGHT_RATIO` / `MIN_MOVEMENT`）は映像に合わせて調整できます。


In [ ]:
from sklearn.cluster import KMeans

# --- フィルタのしきい値（映像に合わせて調整）---
MIN_FRAMES       = max(5, N_TOTAL // 30)   # これ未満しか映らないIDは除外
MIN_HEIGHT_RATIO = 0.06                     # bbox高さが画面高さのこの割合未満は除外（遠い/ベンチ）
MIN_MOVEMENT     = 25.0                      # 総移動量(px)がこれ未満は除外（座ってる人）

def track_stats(dets):
    arr = np.array([[x1, y1, x2, y2] for (_, x1, y1, x2, y2) in dets])
    heights = arr[:, 3] - arr[:, 1]
    cx = (arr[:, 0] + arr[:, 2]) / 2.0
    cy = (arr[:, 1] + arr[:, 3]) / 2.0
    movement = float(np.sum(np.hypot(np.diff(cx), np.diff(cy)))) if len(cx) > 1 else 0.0
    return float(np.median(heights)), movement

valid_ids, feats = [], []
report = []
for tid, dets in track_boxes.items():
    n = len(dets)
    med_h, movement = track_stats(dets)
    ok = (n >= MIN_FRAMES and med_h >= MIN_HEIGHT_RATIO * FRAME_H
          and movement >= MIN_MOVEMENT and len(track_colors[tid]) >= 3)
    report.append((tid, n, round(med_h, 1), round(movement, 1), ok))
    if ok:
        valid_ids.append(tid)
        feats.append(np.median(np.array(track_colors[tid]), axis=0))  # [L,a,b]

print(f"選手候補: {len(valid_ids)} / 全ID {len(track_boxes)} （除外 {len(track_boxes)-len(valid_ids)}）")

team_of = {}
if len(valid_ids) >= 2:
    X = np.array(feats)
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
    for tid, lab in zip(valid_ids, km.labels_):
        team_of[tid] = int(lab)
    # どちらのチームが明るい(白)側か: 平均L
    mean_L = [X[km.labels_ == t][:, 0].mean() for t in (0, 1)]
    bright_team = int(np.argmax(mean_L))
    print(f"✅ チーム分け完了  (チーム{bright_team}が明るい色側)")
    for t in (0, 1):
        members = [i for i in valid_ids if team_of[i] == t]
        print(f"  チーム{t}: {len(members)}人  IDs={members}")
else:
    print("⚠️ 選手が2人以上検出できませんでした。CONFやフィルタ値を調整してください。")

TEAM_COLORS = {0: (0, 0, 255), 1: (255, 128, 0)}  # 赤 / 青(BGR)


### （確認用）フィルタで何が除外されたか

`ok=False` が除外されたID。ベンチ・観客が落とせているか確認し、
本来の選手まで消えていたらしきい値を緩めてください。


In [ ]:
print(f"{'ID':>4} {'frames':>7} {'height':>7} {'move':>7}  keep")
for tid, n, h, mv, ok in sorted(report, key=lambda r: -r[1]):
    print(f"{tid:>4} {n:>7} {h:>7} {mv:>7}  {'✓' if ok else '×'}")


## 5. パス2: 色分け動画の書き出し ＋ CSV出力

動画をディスクから1フレームずつ読み直し、チーム色の枠を描いて書き出します（メモリ一定）。
同時に、各フレームの選手位置を `detections.csv` に保存します（個人成績・戦術分析の土台）。


In [ ]:
# フレーム番号 -> [(tid,x1,y1,x2,y2), ...]（選手として有効なIDのみ）
by_frame = defaultdict(list)
DRAW_NONPLAYERS = False  # True にすると除外IDもグレーで描画
for tid, dets in track_boxes.items():
    if tid not in team_of and not DRAW_NONPLAYERS:
        continue
    for (f_idx, x1, y1, x2, y2) in dets:
        by_frame[f_idx].append((tid, x1, y1, x2, y2))

out_path = "output_team_colored.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (FRAME_W, FRAME_H))
csv_file = open("detections.csv", "w", newline="")
cw = csv.writer(csv_file)
cw.writerow(["frame", "time_s", "track_id", "team", "x1", "y1", "x2", "y2", "cx", "foot_y"])

UNKNOWN = (160, 160, 160)
f_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    for (tid, x1, y1, x2, y2) in by_frame.get(f_idx, []):
        team = team_of.get(tid)
        color = TEAM_COLORS.get(team, UNKNOWN)
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(frame, p1, p2, color, 2)
        label = f"ID{tid}" + (f" T{team}" if team is not None else "")
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (p1[0], p1[1] - th - 6), (p1[0] + tw + 4, p1[1]), color, -1)
        cv2.putText(frame, label, (p1[0] + 2, p1[1] - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        cx = (x1 + x2) / 2.0
        cw.writerow([f_idx, round(f_idx / FPS, 3), tid, team,
                     int(x1), int(y1), int(x2), int(y2), round(cx, 1), int(y2)])
    writer.write(frame)
    f_idx += 1

cap.release(); writer.release(); csv_file.close()
print(f"✅ 書き出し完了: {out_path} / detections.csv ({f_idx}フレーム)")


In [ ]:
# 各選手の要約CSV（出場フレーム数・チーム）
with open("player_tracks.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["track_id", "team", "frames", "seconds"])
    for tid in valid_ids:
        w.writerow([tid, team_of.get(tid), len(track_boxes[tid]),
                    round(len(track_boxes[tid]) / FPS, 1)])
print("✅ player_tracks.csv を書き出しました")


In [ ]:
# Colab上でプレビュー（長い動画は先頭60秒だけ変換して再生）
!ffmpeg -y -t 60 -i output_team_colored.mp4 -vcodec libx264 -movflags +faststart preview.mp4 -loglevel error
from IPython.display import HTML
from base64 import b64encode
data_url = "data:video/mp4;base64," + b64encode(open("preview.mp4", "rb").read()).decode()
HTML(f'<video width="640" controls><source src="{data_url}" type="video/mp4"></video>')


## 6. 結果をダウンロード

In [ ]:
from google.colab import files
for f in ["output_team_colored.mp4", "detections.csv", "player_tracks.csv"]:
    if Path(f).exists():
        files.download(f)


## 7.（任意）個人の精密追跡（SAM2）

記事後半の「1人の選手を最後まで追う」パートです。**ここは重い＆任意ステップ**です。
SAM2は動画を全フレーム先読みするため長尺だとメモリ落ちしますが、**下のセルが対象選手の周辺15秒だけ自動で切り出して**実行するので安全です。
選手検出＋チーム色分け（1〜6）だけで記事のメイン部分は達成なので、ここは飛ばしてもOKです。


In [ ]:
# 追跡したい選手のトラッキングID（上のチーム分け結果から選ぶ）
TARGET_ID = valid_ids[0] if valid_ids else None
print("追跡対象 ID:", TARGET_ID)

start_frame_idx, start_box = None, None
for (f_idx, x1, y1, x2, y2) in sorted(track_boxes[TARGET_ID]):
    start_frame_idx, start_box = f_idx, (x1, y1, x2, y2); break
print("開始フレーム:", start_frame_idx, "初期bbox:", start_box)


In [ ]:
from ultralytics.models.sam import SAM2VideoPredictor

# ★重要★ SAM2は動画を全フレーム先読みするため、長い動画をそのまま渡すとメモリ落ちします。
# ここでは対象選手が最初に現れる位置から SAM2_SECONDS 秒だけ切り出して実行します（OOM防止）。
SAM2_SECONDS = 15
SAM_VIDEO = "sam_clip.mp4"
ss = start_frame_idx / FPS
!ffmpeg -y -ss {ss} -i "{VIDEO_PATH}" -t {SAM2_SECONDS} -c:v libx264 -an "{SAM_VIDEO}" -loglevel error
print(f"✂️ SAM2用に {ss:.1f}s から {SAM2_SECONDS}s を切出し: {SAM_VIDEO}")

# 軽量設定: tinyモデル + imgsz=512。精度重視なら model="sam2.1_b.pt", imgsz=1024
overrides = dict(conf=0.25, task="segment", mode="predict",
                 imgsz=512, model="sam2.1_t.pt", device=DEVICE, verbose=False)
predictor = SAM2VideoPredictor(overrides=overrides)

x1, y1, x2, y2 = start_box
sam_results = predictor(source=SAM_VIDEO, bboxes=[[x1, y1, x2, y2]], labels=[1])
print("✅ SAM2 追跡完了。フレーム数:", len(sam_results))


In [ ]:
sam_out = "output_sam2_track.mp4"
writer2 = cv2.VideoWriter(sam_out, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (FRAME_W, FRAME_H))
overlay_color = np.array([0, 255, 255], dtype=np.uint8)  # 黄色
for res in sam_results:
    base = res.orig_img.copy()
    if res.masks is not None and len(res.masks) > 0:
        m = res.masks.data[0].cpu().numpy().astype(np.uint8)
        if m.shape != base.shape[:2]:
            m = cv2.resize(m, (base.shape[1], base.shape[0]))
        mb = m > 0
        base[mb] = (0.5 * base[mb] + 0.5 * overlay_color).astype(np.uint8)
    writer2.write(base)
writer2.release()
print("✅ 書き出し完了:", sam_out)


In [ ]:
!ffmpeg -y -i output_sam2_track.mp4 -vcodec libx264 -movflags +faststart preview_sam2.mp4 -loglevel error
from IPython.display import HTML
from base64 import b64encode
data_url = "data:video/mp4;base64," + b64encode(open("preview_sam2.mp4", "rb").read()).decode()
HTML(f'<video width="640" controls><source src="{data_url}" type="video/mp4"></video>')


---
## 📝 チューニング＆ロードマップ

**チーム分けが不安定なとき**
- セクション4の `MIN_HEIGHT_RATIO` / `MIN_MOVEMENT` を上げてベンチ・観客をさらに除外
- 確認用セルの表を見て、本来の選手が `×` なら各しきい値を下げる
- ユニフォームが白×薄色で近い場合は `torso_lab` の切り出し範囲を調整

**処理が重い/長い試合**
- 検出モデルを `yolo11n.pt` に（速い）。まずは `TRIM_SECONDS` で短く動作確認

**このノートブックの到達点と、この先（個人成績・戦術分析へ）**

| 段階 | 内容 | 状態 |
|---|---|---|
| ① 逐次処理で長い動画対応 | OOMせず1試合分を処理 | ✅ 本ノート |
| ② チーム分け精度 | LAB色＋ベンチ除外 | ✅ 本ノート |
| ③ 位置データCSV | `detections.csv`（フレーム毎の選手位置） | ✅ 本ノート |
| ④ コートのホモグラフィ | カメラ視点→実寸コート座標（距離・ヒートマップの土台） | ⏭ 次段階 |
| ⑤ 背番号OCR | トラックID→実際の選手番号 | ⏭ 次段階 |
| ⑥ 個人成績 | 走行距離・出場時間・ヒートマップ | ⏭ ④⑤の上に |
| ⑦ 半自動プレー分析 | 種別をマーク→成功/失敗を自動判定→成功率 | ⏭ 研究的・最後 |

コストは Colab 無料枠 + OSSモデルのみ = **0円** で動きます。
